In [1]:
import ipysheet
import numpy as np
from numpy import radians,sin,sqrt,power,average,std
import pandas as pd
import folium
from rich.table import Table
from rich.console import Console

![Random Unsplash Image](./img/ceg1705_logo_notebook.png)


# PRACTICAL 2
## GNSS IN THE FIELD

After returning from the field, you have now to start analysing and reporting your groups results. This notebook will help you to go step by step through the analysis of your data. 

### (a) Coordinate entry and comparison

The sheet below allows you to input your data, enter **your observed coordinates** in the relevant cells (if necessary for your receiver, subtract the antenna height from your ellipsoidal heights before entering the marker height). Remember that longitudes west of Greenwich (i.e. the prime meridian) should be negative (in the deg column only; the notebook has been set up to take care of the rest). If your receiver provided decimals of a minute, leave the sec column with 0 value.

In [4]:
# load true coordinates
lat_true,lon_true,h_true = np.genfromtxt('prac2_true_coords.txt',unpack=True)

# set sites names
site_names = ["TM4","TM5","TM6","TM7","TM8","TM9","TM10","TM11","TM12","TM13"]

In [5]:
sheet = ipysheet.sheet(rows=10, columns=10,
                       column_headers=["Lat.deg", "Lat.min", "Lat.sec",
                                       "Lon.deg","Lon.min","Lon.sec",
                                       "H",
                                       "CQ[plan]","CQ[height]","CQ[3D]"],
                      row_headers=["TM4","TM5","TM6","TM7","TM8","TM9","TM10","TM11","TM12","TM13"])

cells = ipysheet.cell_range(np.zeros((10,10)),row_start=0, column_start=0)

sheet

Sheet(cells=(Cell(column_end=9, column_start=0, row_end=9, row_start=0, squeeze_column=False, squeeze_row=Fals…

N.B 

- Cells with "👇 ✍🏻" indicate that you have to input your answer in the following cell.
- Cells with "✏️ Your answer here" where you must write your comments.

In [6]:
data = ipysheet.to_dataframe(sheet)

Let's convert you coordinates to decimal degree. We prepared a function to do this.

In [7]:
# Latitude
sign = np.sign(data['Lat.deg'])
lat_dec = data['Lat.deg'] + sign*data['Lat.min']/60 + sign*data['Lat.sec']/3600

# Longitude
sign = np.sign(data['Lon.deg'])
lon_dec = data['Lon.deg'] + sign*data['Lon.min']/60 + sign*data['Lon.sec']/3600

# Height
h = data['H']

Now Let's plot your coordinates to check on map 😜😜

In [8]:
# Create interactive map with default basemap
map_moor = folium.Map(location=[54.98488, -1.618106],zoom_start=17)
for i in range(len(lat_dec)):
    folium.Marker( location=[ lat_dec[i], lon_dec[i] ],popup=site_names[i], fill_color='#43d9de', radius=8 ).add_to( map_moor )

map_moor

The precise surveyed coordinates are given in the variables `lat_true`, `lon_true` and `h_true`. You must compute the difference (observed minus true) **in metres**.


Let's first compute the difference in degrees and store in the variables `diff_lat` and `diff_lon`.

To do an array difference you simply subtract; For example for computing latitude differences:        
```
    diff_lat = lat_dec - lat_true
```

👇 ✍🏻

In [ ]:
diff_lat = 
diff_lon = 
diff_h_meters = 

The **difference in metres** is the **difference in radians** times the radius of curvature in the meridian ($\nu$).

<div class="alert alert-block alert-info">

To convert from degrees to radians, we use the function `radians` from the module numpy.
</div>

e.g 

    lat_radians = np.radians(lat_degrees)


👇 ✍🏻

In [ ]:
diff_lat_rad = 

The radius of curvature $\nu$ (`nu`) is :

In [ ]:
a = 6378137.000
b = 6356752.314
e_2 = (a**2-b**2)/a**2
phi_A = np.average(np.radians(lat_true))
nu = a/np.sqrt(1-e_2*np.sin(phi_A)**2)

Now, we're ready to calculate the latitude differences in meters. We store then in the variables `diff_lat_meters`.


👇 ✍🏻

In [ ]:
diff_lat_meters = 

For the longitude difference, make use of the longitude in radians. First, we convert the differences from degress to radians:


👇 ✍🏻

In [ ]:
diff_lon_rad = 

The difference in metres is the difference in radians, times the radius of curvature in the prime vertical ($\rho$ or `rho`), times the cosine of the latitude $\phi_A$.  The combined quantity $\rho cos(\phi_A)$ is computed for you in the cell below.  

In [ ]:
rho = a*(1-e_2)/np.power(1-e_2*np.sin(phi_A)**2,1.5)

The longitude differences in meters:


👇 ✍🏻

In [ ]:
diff_lon_meters = 

In [ ]:
table = Table(title="Differences Lat. Lon. in meters")
table.add_column("Diff. Lat.[m]")
table.add_column("Diff. Lon.[m]")
table.add_column("Diff. height[m]")
table.add_row(str(diff_lat_meters),str(diff_lon_meters),str(diff_h_meters))
console = Console()
console.print(table)

You should now have a set of numbers giving the distances between your observed coordinates and the true ones.  Expect these differences to be anywhere between a few millimetres and a few tens of metres in size.  If your receiver could not observe a particular coordinate, do not compute a coordinate difference (leave the relevant cells blank).  If there are any differences which are much larger than the others, check your data and if necessary discard these outliers.

### (a) Statistics of coordinate differences

Compute the mean coordinate differences $\mu$ in latitude, longitude and height. For this you need the function numpy `average()` which computes the arithmetic mean. These values represent the accuracy (bias or systematic error) of your receiver today, i.e. the average offset between your results and the truth.

e.g of using the average function:

    avg_diff_lat = average(diff_lat_meters)
    


👇 ✍🏻

In [ ]:
avg_diff_lat = 
avg_diff_lon = 
avg_diff_h = 

Compute the standard deviations $\sigma$ of the coordinate differences in latitude, longitude and height.  For this you need the function `std()` which computes the standard deviation of the values in the specified array, for example:

        std_diff_lat = std(diff_lat_meters)   
    
    
These values represent the precision (random error) of your receiver today, i.e. the scatter of your results about their systematic error (mean).


👇 ✍🏻

In [ ]:
std_diff_lat = 
std_diff_lon = 
std_diff_h = 

Compute the root mean squares (RMS) of the coordinate differences in latitude, longitude and height, given by the formula : $\sqrt{\mu^2 + \sigma^2}$ . This is an overall representation of the coordinate quality (accuracy and precision combined).


👇 ✍🏻

In [ ]:
rms_diff_lat = 
rms_diff_lon = 
rms_diff_h = 

In [ ]:
table = Table(title="Statistics of differences")
table.add_column("Coord.")
table.add_column("avg(m)")
table.add_column("std(m)")
table.add_column("rms(m)")
table.add_row("Latitude",str(avg_diff_lat),str(std_diff_lat),str(rms_diff_lat))
table.add_row("Longitude",str(avg_diff_lon),str(std_diff_lon),str(rms_diff_lon))
table.add_row("Height",str(avg_diff_h),str(std_diff_h),str(rms_diff_h))
console = Console()
console.print(table)

You're done. Time to submit your work 👉 Canvas.